In [5]:
# Cella Jupyter TDCS-MI
import os
import mne
import pandas as pd
import numpy as np
from scipy.stats import skew, kurtosis
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

# --- CONFIG ---
dataset_path = "/Volumes/KINGSTON/NeuroCore/neurocore_lab/data/EEG/TDCS_ds006126"
output_csv = os.path.join(dataset_path, "TDCS_MI_signature.csv")
output_pdf = os.path.join(dataset_path, "TDCS_MI_report.pdf")
freq_bands = {'delta': (1,4), 'theta': (4,8), 'alpha': (8,13), 'beta': (13,30), 'gamma': (30,45)}

# --- TROVA METADATA (txt o tsv) ---
participants_file = None
session_file = None

for fname in os.listdir(dataset_path):
    if 'participants' in fname.lower() and fname.endswith(('.tsv', '.txt')):
        participants_file = os.path.join(dataset_path, fname)
    if 'session' in fname.lower() and fname.endswith(('.tsv', '.txt')):
        session_file = os.path.join(dataset_path, fname)

if participants_file is None or session_file is None:
    raise FileNotFoundError("Non sono stati trovati participants.txt/tsv o session.txt/tsv nella cartella dataset.")

# --- LETTURA ROBUSTA TXT/TSV ---
def robust_read_csv(file_path):
    encodings = ['utf-8', 'latin1', 'cp1252']
    for enc in encodings:
        try:
            return pd.read_csv(file_path, sep='\t', encoding=enc)
        except Exception:
            continue
    raise UnicodeDecodeError(f"Impossibile leggere il file {file_path} con codifica comune.")

participants = robust_read_csv(participants_file)
sessions = robust_read_csv(session_file)

# --- INIZIALIZZAZIONE CSV ---
csv_rows = []

# --- INIZIALIZZAZIONE PDF ---
pdf = PdfPages(output_pdf)

# --- SCANSIONE SOGGETTI ---
for subj_folder in sorted(os.listdir(dataset_path)):
    subj_path = os.path.join(dataset_path, subj_folder)
    if not os.path.isdir(subj_path):
        continue
    
    vhdr_files = [f for f in os.listdir(subj_path) if f.endswith('.vhdr')]
    if len(vhdr_files) == 0:
        print(f"Skip {subj_folder}: nessun .vhdr")
        continue
    
    for vhdr_file in vhdr_files:
        vhdr_path = os.path.join(subj_path, vhdr_file)
        print(f"Lettura: {vhdr_path}")
        raw = mne.io.read_raw_brainvision(vhdr_path, preload=True, verbose=False)
        raw.filter(1, 45, fir_design='firwin', verbose=False)
        
        events, event_id = mne.events_from_annotations(raw, verbose=False)
        for cond_name, cond_code in event_id.items():
            epochs = mne.Epochs(raw, events, event_id={cond_name: cond_code}, tmin=0, tmax=2,
                                baseline=None, preload=True, verbose=False)
            data = epochs.get_data()
            n_epochs, n_ch, n_times = data.shape
            
            for ch_idx, ch_name in enumerate(raw.info['ch_names']):
                signal = data[:, ch_idx, :].reshape(-1)
                row = {
                    'subject': subj_folder,
                    'condition': cond_name,
                    'channel': ch_name,
                    'mean': np.mean(signal),
                    'variance': np.var(signal),
                    'skewness': skew(signal),
                    'kurtosis': kurtosis(signal)
                }
                psd, freqs = mne.time_frequency.psd_array_welch(signal, sfreq=raw.info['sfreq'], fmin=1, fmax=45, verbose=False)
                for band, (fmin, fmax) in freq_bands.items():
                    idx = np.logical_and(freqs >= fmin, freqs <= fmax)
                    row[f'power_{band}'] = np.mean(psd[idx])
                csv_rows.append(row)
            
            # PDF: topografia semplice
            avg_signal = data.mean(axis=0)
            psd_avg, freqs = mne.time_frequency.psd_array_welch(avg_signal, sfreq=raw.info['sfreq'], fmin=1, fmax=45, verbose=False)
            fig, ax = plt.subplots()
            im = ax.imshow(psd_avg.mean(axis=1).reshape(-1,1), aspect='auto', cmap='viridis')
            ax.set_title(f"{subj_folder} - {cond_name} PSD mean")
            plt.colorbar(im, ax=ax, label='PSD')
            pdf.savefig(fig)
            plt.close(fig)

# --- SALVA CSV E PDF ---
df = pd.DataFrame(csv_rows)
df.to_csv(output_csv, index=False)
pdf.close()

print(f"✅ CSV salvato in: {output_csv}")
print(f"✅ PDF salvato in: {output_pdf}")

Skip sub-AnSt1: nessun .vhdr
Skip sub-FeKl03: nessun .vhdr
Skip sub-IrCh04: nessun .vhdr
Skip sub-Kiko09: nessun .vhdr
Skip sub-SoNi11: nessun .vhdr
✅ CSV salvato in: /Volumes/KINGSTON/NeuroCore/neurocore_lab/data/EEG/TDCS_ds006126/TDCS_MI_signature.csv
✅ PDF salvato in: /Volumes/KINGSTON/NeuroCore/neurocore_lab/data/EEG/TDCS_ds006126/TDCS_MI_report.pdf


In [6]:
# Cella Jupyter: "TDCS-MI"
import os
import mne
import pandas as pd
import numpy as np
from scipy.stats import skew, kurtosis
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

# --- CONFIG ---
dataset_path = "/Volumes/KINGSTON/NeuroCore/neurocore_lab/data/EEG/TDCS_ds006126"
output_csv = os.path.join(dataset_path, "TDCS_MI_superfirma.csv")
output_pdf = os.path.join(dataset_path, "TDCS_MI_superfirma_report.pdf")
freq_bands = {'delta': (1,4), 'theta': (4,8), 'alpha': (8,13), 'beta': (13,30), 'gamma': (30,45)}

# --- TROVA METADATA (txt o tsv) ---
participants_file = None
session_file = None

for fname in os.listdir(dataset_path):
    if 'participants' in fname.lower() and fname.endswith(('.tsv', '.txt')):
        participants_file = os.path.join(dataset_path, fname)
    if 'session' in fname.lower() and fname.endswith(('.tsv', '.txt')):
        session_file = os.path.join(dataset_path, fname)

if participants_file is None or session_file is None:
    raise FileNotFoundError("Non sono stati trovati participants.txt/tsv o session.txt/tsv nella cartella dataset.")

# --- LETTURA ROBUSTA TXT/TSV ---
def robust_read_csv(file_path):
    for enc in ['utf-8', 'latin1', 'cp1252']:
        try:
            return pd.read_csv(file_path, sep='\t', encoding=enc)
        except Exception:
            continue
    raise UnicodeDecodeError(f"Impossibile leggere il file {file_path} con codifica comune.")

participants = robust_read_csv(participants_file)
sessions = robust_read_csv(session_file)

# --- INIZIALIZZAZIONE CSV e PDF ---
csv_rows = []
pdf = PdfPages(output_pdf)

# --- SCANSIONE SOGGETTI ---
for subj_folder in sorted(os.listdir(dataset_path)):
    subj_path = os.path.join(dataset_path, subj_folder)
    if not os.path.isdir(subj_path):
        continue
    
    vhdr_files = [f for f in os.listdir(subj_path) if f.endswith('.vhdr')]
    if len(vhdr_files) == 0:
        print(f"Skip {subj_folder}: nessun .vhdr")
        continue
    
    for vhdr_file in vhdr_files:
        vhdr_path = os.path.join(subj_path, vhdr_file)
        print(f"Lettura: {vhdr_path}")
        raw = mne.io.read_raw_brainvision(vhdr_path, preload=True, verbose=False)
        raw.filter(1, 45, fir_design='firwin', verbose=False)
        
        events, event_id = mne.events_from_annotations(raw, verbose=False)
        for cond_name, cond_code in event_id.items():
            epochs = mne.Epochs(raw, events, event_id={cond_name: cond_code}, tmin=0, tmax=2,
                                baseline=None, preload=True, verbose=False)
            data = epochs.get_data()
            n_epochs, n_ch, n_times = data.shape
            
            for ch_idx, ch_name in enumerate(raw.info['ch_names']):
                signal_epochs = data[:, ch_idx, :]
                signal_flat = signal_epochs.reshape(-1)
                
                # --- TIME-DOMAIN FEATURES ---
                mean_val = np.mean(signal_flat)
                var_val = np.var(signal_flat)
                skew_val = skew(signal_flat)
                kurt_val = kurtosis(signal_flat)
                
                # --- FREQUENCY-DOMAIN (PSD) ---
                psd, freqs = mne.time_frequency.psd_array_welch(signal_flat, sfreq=raw.info['sfreq'], fmin=1, fmax=45, verbose=False)
                band_power = {}
                for band, (fmin, fmax) in freq_bands.items():
                    idx = np.logical_and(freqs >= fmin, freqs <= fmax)
                    band_power[band] = np.mean(psd[idx])
                
                # --- PERSISTENZA TEMPORALE ---
                # Correlazione media tra epoche successive
                if n_epochs > 1:
                    corr_matrix = np.corrcoef(signal_epochs)
                    persistence = np.mean(np.triu(corr_matrix, k=1))
                else:
                    persistence = np.nan
                
                # --- RESILIENZA FUNZIONALE ---
                resilience = 1 / (1 + np.var(signal_flat))  # più stabile = più resiliente
                
                # --- COSTRUZIONE CSV ROW ---
                row = {
                    'subject': subj_folder,
                    'condition': cond_name,
                    'channel': ch_name,
                    'mean': mean_val,
                    'variance': var_val,
                    'skewness': skew_val,
                    'kurtosis': kurt_val,
                    'persistence': persistence,
                    'resilience': resilience
                }
                for band in freq_bands:
                    row[f'power_{band}'] = band_power[band]
                csv_rows.append(row)
            
            # --- TOPOGRAPHY PDF ---
            avg_signal = signal_epochs.mean(axis=0)
            psd_avg, freqs = mne.time_frequency.psd_array_welch(avg_signal, sfreq=raw.info['sfreq'], fmin=1, fmax=45, verbose=False)
            fig, ax = plt.subplots(figsize=(6,8))
            im = ax.imshow(psd_avg.mean(axis=1).reshape(-1,1), aspect='auto', cmap='viridis')
            ax.set_title(f"{subj_folder} - {cond_name} PSD mean")
            plt.colorbar(im, ax=ax, label='PSD')
            pdf.savefig(fig)
            plt.close(fig)

# --- SALVA CSV E PDF ---
df = pd.DataFrame(csv_rows)
df.to_csv(output_csv, index=False)
pdf.close()

print(f"✅ Superfirma CSV salvata in: {output_csv}")
print(f"✅ Superfirma PDF salvata in: {output_pdf}")

Skip Output: nessun .vhdr
Skip sub-AnSt1: nessun .vhdr
Skip sub-FeKl03: nessun .vhdr
Skip sub-IrCh04: nessun .vhdr
Skip sub-Kiko09: nessun .vhdr
Skip sub-SoNi11: nessun .vhdr
✅ Superfirma CSV salvata in: /Volumes/KINGSTON/NeuroCore/neurocore_lab/data/EEG/TDCS_ds006126/TDCS_MI_superfirma.csv
✅ Superfirma PDF salvata in: /Volumes/KINGSTON/NeuroCore/neurocore_lab/data/EEG/TDCS_ds006126/TDCS_MI_superfirma_report.pdf


In [8]:
# TDCS-MI”
import os, mne, pandas as pd, numpy as np
from scipy.stats import skew, kurtosis
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

dataset_path = "/Volumes/KINGSTON/NeuroCore/neurocore_lab/data/EEG/TDCS_ds006126"
output_csv = os.path.join("/Users/Davide/Desktop", "TDCS_MI_superfirma.csv")  # salva su desktop
output_pdf = os.path.join("/Users/Davide/Desktop", "TDCS_MI_superfirma_report.pdf")
freq_bands = {'delta': (1,4), 'theta': (4,8), 'alpha': (8,13), 'beta': (13,30), 'gamma': (30,45)}

# --- metadata txt/tsv ---
participants_file, session_file = None, None
for f in os.listdir(dataset_path):
    if 'participants' in f.lower() and f.endswith(('.tsv','.txt')): participants_file=os.path.join(dataset_path,f)
    if 'session' in f.lower() and f.endswith(('.tsv','.txt')): session_file=os.path.join(dataset_path,f)
if participants_file is None or session_file is None: raise FileNotFoundError("participants/session non trovati")

def robust_read_csv(fp):
    for enc in ['utf-8','latin1','cp1252']:
        try: return pd.read_csv(fp, sep='\t', encoding=enc)
        except: continue
    raise UnicodeDecodeError(f"Impossibile leggere {fp}")

participants = robust_read_csv(participants_file)
sessions = robust_read_csv(session_file)

csv_rows = []
pdf = PdfPages(output_pdf)

for subj_folder in sorted(os.listdir(dataset_path)):
    subj_path = os.path.join(dataset_path, subj_folder)
    if not os.path.isdir(subj_path): continue
    vhdr_files = [f for f in os.listdir(subj_path) if f.endswith('.vhdr')]
    if not vhdr_files: continue
    
    for vhdr_file in vhdr_files:
        raw = mne.io.read_raw_brainvision(os.path.join(subj_path,vhdr_file), preload=True, verbose=False)
        raw.filter(1,45,fir_design='firwin', verbose=False)
        events, event_id = mne.events_from_annotations(raw, verbose=False)
        
        for cond_name, cond_code in event_id.items():
            epochs = mne.Epochs(raw, events, event_id={cond_name:cond_code}, tmin=0, tmax=2,
                                baseline=None, preload=True, verbose=False)
            data = epochs.get_data()
            if data.size==0: continue
            n_epochs, n_ch, n_times = data.shape
            
            for ch_idx, ch_name in enumerate(raw.info['ch_names']):
                sig_epochs = data[:,ch_idx,:]
                sig_flat = sig_epochs.reshape(-1)
                
                # time-domain
                mean_val, var_val = np.mean(sig_flat), np.var(sig_flat)
                skew_val, kurt_val = skew(sig_flat), kurtosis(sig_flat)
                # frequency-domain
                psd, freqs = mne.time_frequency.psd_array_welch(sig_flat, sfreq=raw.info['sfreq'], fmin=1, fmax=45, verbose=False)
                band_power = {b: np.mean(psd[(freqs>=fmin)&(freqs<=fmax)]) for b,(fmin,fmax) in freq_bands.items()}
                # persistenza e resilienza
                persistence = np.mean(np.corrcoef(sig_epochs)) if n_epochs>1 else np.nan
                resilience = 1/(1+var_val)
                
                row={'subject':subj_folder,'condition':cond_name,'channel':ch_name,
                     'mean':mean_val,'variance':var_val,'skewness':skew_val,'kurtosis':kurt_val,
                     'persistence':persistence,'resilience':resilience}
                row.update({f'power_{b}':band_power[b] for b in freq_bands})
                csv_rows.append(row)
            
            # --- PDF robusto ---
            try:
                avg_sig = sig_epochs.mean(axis=0)
                psd_avg, _ = mne.time_frequency.psd_array_welch(avg_sig, sfreq=raw.info['sfreq'], fmin=1, fmax=45, verbose=False)
                if psd_avg.size>0:
                    fig,ax = plt.subplots(figsize=(6,8))
                    ax.imshow(psd_avg.reshape(-1,1),aspect='auto',cmap='viridis')
                    ax.set_title(f"{subj_folder}-{cond_name} PSD mean")
                    plt.colorbar(ax=ax,label='PSD')
                    pdf.savefig(fig)
                    plt.close(fig)
            except Exception as e:
                print(f"PDF skip {subj_folder}-{cond_name}: {e}")

# --- salva tutto ---
df = pd.DataFrame(csv_rows)
df.to_csv(output_csv,index=False)
pdf.close()
print(f"✅ CSV salvato in {output_csv}")
print(f"✅ PDF salvato in {output_pdf}")

✅ CSV salvato in /Users/Davide/Desktop/TDCS_MI_superfirma.csv
✅ PDF salvato in /Users/Davide/Desktop/TDCS_MI_superfirma_report.pdf


In [9]:
import os
import mne
import pandas as pd
import numpy as np
from scipy.stats import skew, kurtosis
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

# --- CONFIG ---
dataset_path = "/Volumes/KINGSTON/NeuroCore/neurocore_lab/data/EEG/TDCS_ds006126"
output_folder = os.path.join(dataset_path, "Output")
os.makedirs(output_folder, exist_ok=True)  # crea Output se non esiste

output_csv = os.path.join(output_folder, "TDCS_MI_superfirma.csv")
output_pdf = os.path.join(output_folder, "TDCS_MI_superfirma_report.pdf")
freq_bands = {'delta': (1,4), 'theta': (4,8), 'alpha': (8,13), 'beta': (13,30), 'gamma': (30,45)}

# --- TROVA METADATA (txt o tsv) ---
participants_file = None
session_file = None
for f in os.listdir(dataset_path):
    if 'participants' in f.lower() and f.endswith(('.tsv','.txt')): participants_file=os.path.join(dataset_path,f)
    if 'session' in f.lower() and f.endswith(('.tsv','.txt')): session_file=os.path.join(dataset_path,f)
if participants_file is None or session_file is None: raise FileNotFoundError("participants/session non trovati")

def robust_read_csv(fp):
    for enc in ['utf-8','latin1','cp1252']:
        try: return pd.read_csv(fp, sep='\t', encoding=enc)
        except: continue
    raise UnicodeDecodeError(f"Impossibile leggere {fp}")

participants = robust_read_csv(participants_file)
sessions = robust_read_csv(session_file)

csv_rows = []
pdf = PdfPages(output_pdf)

# --- SCANSIONE SOGGETTI ---
for subj_folder in sorted(os.listdir(dataset_path)):
    subj_path = os.path.join(dataset_path, subj_folder)
    if not os.path.isdir(subj_path): continue
    vhdr_files = [f for f in os.listdir(subj_path) if f.endswith('.vhdr')]
    if not vhdr_files: continue

    for vhdr_file in vhdr_files:
        raw = mne.io.read_raw_brainvision(os.path.join(subj_path,vhdr_file), preload=True, verbose=False)
        raw.filter(1,45,fir_design='firwin', verbose=False)
        events, event_id = mne.events_from_annotations(raw, verbose=False)

        for cond_name, cond_code in event_id.items():
            epochs = mne.Epochs(raw, events, event_id={cond_name:cond_code}, tmin=0, tmax=2,
                                baseline=None, preload=True, verbose=False)
            data = epochs.get_data()
            if data.size==0: continue
            n_epochs, n_ch, n_times = data.shape

            for ch_idx, ch_name in enumerate(raw.info['ch_names']):
                sig_epochs = data[:,ch_idx,:]
                sig_flat = sig_epochs.reshape(-1)
                mean_val, var_val = np.mean(sig_flat), np.var(sig_flat)
                skew_val, kurt_val = skew(sig_flat), kurtosis(sig_flat)
                psd, freqs = mne.time_frequency.psd_array_welch(sig_flat, sfreq=raw.info['sfreq'], fmin=1, fmax=45, verbose=False)
                band_power = {b: np.mean(psd[(freqs>=fmin)&(freqs<=fmax)]) for b,(fmin,fmax) in freq_bands.items()}
                persistence = np.mean(np.corrcoef(sig_epochs)) if n_epochs>1 else np.nan
                resilience = 1/(1+var_val)

                row={'subject':subj_folder,'condition':cond_name,'channel':ch_name,
                     'mean':mean_val,'variance':var_val,'skewness':skew_val,'kurtosis':kurt_val,
                     'persistence':persistence,'resilience':resilience}
                row.update({f'power_{b}':band_power[b] for b in freq_bands})
                csv_rows.append(row)

            # --- PDF robusto ---
            try:
                avg_sig = sig_epochs.mean(axis=0)
                psd_avg, _ = mne.time_frequency.psd_array_welch(avg_sig, sfreq=raw.info['sfreq'], fmin=1, fmax=45, verbose=False)
                if psd_avg.size>0:
                    fig,ax = plt.subplots(figsize=(6,8))
                    ax.imshow(psd_avg.reshape(-1,1),aspect='auto',cmap='viridis')
                    ax.set_title(f"{subj_folder}-{cond_name} PSD mean")
                    plt.colorbar(ax=ax,label='PSD')
                    pdf.savefig(fig)
                    plt.close(fig)
            except Exception as e:
                print(f"PDF skip {subj_folder}-{cond_name}: {e}")

# --- SALVA CSV E PDF ---
df = pd.DataFrame(csv_rows)
df.to_csv(output_csv,index=False)
pdf.close()

print(f"✅ CSV salvato in: {output_csv}")
print(f"✅ PDF salvato in: {output_pdf}")

✅ CSV salvato in: /Volumes/KINGSTON/NeuroCore/neurocore_lab/data/EEG/TDCS_ds006126/Output/TDCS_MI_superfirma.csv
✅ PDF salvato in: /Volumes/KINGSTON/NeuroCore/neurocore_lab/data/EEG/TDCS_ds006126/Output/TDCS_MI_superfirma_report.pdf


In [10]:
# Cella avanzata TDCS-MI
import os
import mne
import pandas as pd
import numpy as np
from scipy.stats import skew, kurtosis
import itertools

# --- CONFIG ---
dataset_path = "/Volumes/KINGSTON/NeuroCore/neurocore_lab/data/EEG/TDCS_ds006126"
output_folder = os.path.join(dataset_path, "Output")
os.makedirs(output_folder, exist_ok=True)

output_csv = os.path.join(output_folder, "TDCS_MI_superfirma_multidim.csv")
freq_bands = {'delta': (1,4), 'theta': (4,8), 'alpha': (8,13), 'beta': (13,30), 'gamma': (30,45)}
coherence_pairs = [('Fz','Cz'), ('Pz','Oz'), ('C3','C4')]  # esempio coppie chiave

# --- METADATA ---
participants_file = None
session_file = None
for f in os.listdir(dataset_path):
    if 'participants' in f.lower() and f.endswith(('.tsv','.txt')): participants_file=os.path.join(dataset_path,f)
    if 'session' in f.lower() and f.endswith(('.tsv','.txt')): session_file=os.path.join(dataset_path,f)
if participants_file is None or session_file is None: raise FileNotFoundError("participants/session non trovati")

def robust_read_csv(fp):
    for enc in ['utf-8','latin1','cp1252']:
        try: return pd.read_csv(fp, sep='\t', encoding=enc)
        except: continue
    raise UnicodeDecodeError(f"Impossibile leggere {fp}")

participants = robust_read_csv(participants_file)
sessions = robust_read_csv(session_file)

csv_rows = []

# --- SCANSIONE SOGGETTI ---
for subj_folder in sorted(os.listdir(dataset_path)):
    subj_path = os.path.join(dataset_path, subj_folder)
    if not os.path.isdir(subj_path): continue
    vhdr_files = [f for f in os.listdir(subj_path) if f.endswith('.vhdr')]
    if not vhdr_files: continue

    for vhdr_file in vhdr_files:
        raw = mne.io.read_raw_brainvision(os.path.join(subj_path,vhdr_file), preload=True, verbose=False)
        raw.filter(1,45,fir_design='firwin', verbose=False)
        events, event_id = mne.events_from_annotations(raw, verbose=False)

        for cond_name, cond_code in event_id.items():
            epochs = mne.Epochs(raw, events, event_id={cond_name:cond_code}, tmin=0, tmax=2,
                                baseline=None, preload=True, verbose=False)
            data = epochs.get_data()
            if data.size==0: continue
            n_epochs, n_ch, n_times = data.shape
            ch_names = raw.info['ch_names']

            # --- PER OGNI CANALE ---
            for ch_idx, ch_name in enumerate(ch_names):
                sig_epochs = data[:,ch_idx,:]
                sig_flat = sig_epochs.reshape(-1)

                # PSD bande
                psd, freqs = mne.time_frequency.psd_array_welch(sig_flat, sfreq=raw.info['sfreq'], fmin=1, fmax=45, verbose=False)
                band_power = {b: np.mean(psd[(freqs>=fmin)&(freqs<=fmax)]) for b,(fmin,fmax) in freq_bands.items()}
                # Ratios
                alpha_beta_ratio = band_power['alpha']/band_power['beta'] if band_power['beta']>0 else np.nan
                gamma_alpha_ratio = band_power['gamma']/band_power['alpha'] if band_power['alpha']>0 else np.nan
                # Persistenza
                persistence = np.mean(np.corrcoef(sig_epochs)) if n_epochs>1 else np.nan
                # Variabilità PSD tra epoche
                psd_epochs = np.array([np.mean(mne.time_frequency.psd_array_welch(sig_epochs[i,:], sfreq=raw.info['sfreq'], fmin=1, fmax=45, verbose=False)[0]) for i in range(n_epochs)])
                psd_var = np.var(psd_epochs)
                psd_skew = skew(psd_epochs)
                psd_kurt = kurtosis(psd_epochs)

                row={'subject':subj_folder,'condition':cond_name,'channel':ch_name,
                     'persistence':persistence,'psd_var':psd_var,'psd_skew':psd_skew,'psd_kurt':psd_kurt,
                     'alpha_beta_ratio':alpha_beta_ratio,'gamma_alpha_ratio':gamma_alpha_ratio}
                row.update({f'power_{b}':band_power[b] for b in freq_bands})
                csv_rows.append(row)

            # --- COERENZA TRA CANALI SELEZIONATI ---
            for ch1,ch2 in coherence_pairs:
                if ch1 in ch_names and ch2 in ch_names:
                    idx1, idx2 = ch_names.index(ch1), ch_names.index(ch2)
                    sig1, sig2 = data[:,idx1,:].reshape(-1), data[:,idx2,:].reshape(-1)
                    # coerenza semplice (correlazione)
                    coh = np.corrcoef(sig1,sig2)[0,1]
                    csv_rows.append({'subject':subj_folder,'condition':cond_name,'channel_pair':f'{ch1}-{ch2}','coherence':coh})

# --- SALVA CSV ---
df = pd.DataFrame(csv_rows)
df.to_csv(output_csv,index=False)
print(f"✅ Superfirma multidimensionale salvata in: {output_csv}")

✅ Superfirma multidimensionale salvata in: /Volumes/KINGSTON/NeuroCore/neurocore_lab/data/EEG/TDCS_ds006126/Output/TDCS_MI_superfirma_multidim.csv


In [12]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# --- CONFIG ---
dataset_path = "/Volumes/KINGSTON/NeuroCore/neurocore_lab/data/EEG/TDCS_ds006126/Output"
csv_file = os.path.join(dataset_path, "TDCS_MI_superfirma_multidim.csv")

# --- LETTURA CSV ---
df = pd.read_csv(csv_file)

# --- AGGREGAZIONE FEATURE PER SOGGETTO/CONDIZIONE ---
# Raggruppiamo tutte le feature numeriche per soggetto e condizione
feature_cols = [c for c in df.columns if c not in ['subject','condition','channel','channel_pair']]
df_features = df.groupby(['subject','condition'])[feature_cols].mean().reset_index()

# --- DEFINIZIONE TARGET ---
# Esempio: se vuoi predire TDCS ON vs OFF
# Sostituisci i valori esatti delle tue condizioni
# Ad esempio: condizione == "TDCS_ON" -> 1, else -> 0
df_features['target'] = df_features['condition'].apply(lambda x: 1 if "ON" in x.upper() else 0)

# --- MATRICE FEATURE e TARGET ---
X = df_features[feature_cols].values
y = df_features['target'].values

# --- CROSS-VALIDATED AUC ---
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
aucs = []
for train_idx, test_idx in cv.split(X, y):
    clf = RandomForestClassifier(n_estimators=200, random_state=42)
    clf.fit(X[train_idx], y[train_idx])
    probs = clf.predict_proba(X[test_idx])[:,1]
    aucs.append(roc_auc_score(y[test_idx], probs))

print(f"✅ AUC medio: {np.mean(aucs):.4f}")

EmptyDataError: No columns to parse from file

In [14]:
# Se separato da tab
df = pd.read_csv(csv_file, sep='\t', engine='python', encoding='utf-8')

# Se separato da punto e virgola
df = pd.read_csv(csv_file, sep=';', engine='python', encoding='utf-8')

EmptyDataError: No columns to parse from file

In [15]:
import os

csv_file = "/Volumes/KINGSTON/NeuroCore/neurocore_lab/data/EEG/TDCS_ds006126/Output/TDCS_MI_superfirma_multidim.csv"
print("File exists:", os.path.exists(csv_file))
print("File size (bytes):", os.path.getsize(csv_file))

with open(csv_file,'r',encoding='utf-8',errors='ignore') as f:
    lines = f.readlines()
    print("Prime 5 righe:")
    for line in lines[:5]:
        print(line.strip())

File exists: True
File size (bytes): 1
Prime 5 righe:



In [16]:
import os
import mne
import pandas as pd
import numpy as np

dataset_path = "/Volumes/KINGSTON/NeuroCore/neurocore_lab/data/EEG/TDCS_ds006126"
output_folder = os.path.join(dataset_path, "Output")
os.makedirs(output_folder, exist_ok=True)
output_csv = os.path.join(output_folder, "TDCS_MI_superfirma_debug.csv")

csv_rows = []

print("=== DEBUG GENERAZIONE CSV TDCS-MI ===")

# Scansione soggetti
for subj_folder in sorted(os.listdir(dataset_path)):
    subj_path = os.path.join(dataset_path, subj_folder)
    if not os.path.isdir(subj_path):
        continue
    vhdr_files = [f for f in os.listdir(subj_path) if f.endswith('.vhdr')]
    if not vhdr_files:
        print(f"[WARN] {subj_folder} -> nessun file .vhdr trovato")
        continue

    for vhdr_file in vhdr_files:
        raw_path = os.path.join(subj_path, vhdr_file)
        try:
            raw = mne.io.read_raw_brainvision(raw_path, preload=True, verbose=False)
        except Exception as e:
            print(f"[ERROR] {vhdr_file}: {e}")
            continue
        raw.filter(1,45,fir_design='firwin', verbose=False)
        events, event_id = mne.events_from_annotations(raw, verbose=False)

        for cond_name, cond_code in event_id.items():
            epochs = mne.Epochs(raw, events, event_id={cond_name:cond_code}, tmin=0, tmax=2,
                                baseline=None, preload=True, verbose=False)
            data = epochs.get_data()
            n_epochs, n_ch, n_times = data.shape if data.size>0 else (0,0,0)
            
            if n_epochs==0:
                print(f"[SKIP] {subj_folder} - {cond_name} -> nessuna epoca valida")
                continue

            print(f"[OK] {subj_folder} - {cond_name} -> epoche: {n_epochs}, canali: {n_ch}, punti temporali: {n_times}")

            # Scrive almeno qualche feature semplice come test
            for ch_idx, ch_name in enumerate(raw.info['ch_names']):
                sig_epochs = data[:,ch_idx,:]
                sig_flat = sig_epochs.reshape(-1)
                mean_val = np.mean(sig_flat)
                var_val = np.var(sig_flat)
                csv_rows.append({'subject':subj_folder,'condition':cond_name,'channel':ch_name,
                                 'mean':mean_val,'variance':var_val})

# Salvataggio CSV debug
if csv_rows:
    df_debug = pd.DataFrame(csv_rows)
    df_debug.to_csv(output_csv,index=False)
    print(f"✅ CSV debug salvato in: {output_csv}")
else:
    print("❌ Nessuna epoca valida trovata in tutto il dataset, CSV vuoto")

=== DEBUG GENERAZIONE CSV TDCS-MI ===
[WARN] Output -> nessun file .vhdr trovato
[WARN] sub-AnSt1 -> nessun file .vhdr trovato
[WARN] sub-FeKl03 -> nessun file .vhdr trovato
[WARN] sub-IrCh04 -> nessun file .vhdr trovato
[WARN] sub-Kiko09 -> nessun file .vhdr trovato
[WARN] sub-SoNi11 -> nessun file .vhdr trovato
❌ Nessuna epoca valida trovata in tutto il dataset, CSV vuoto


In [17]:
import os
import mne
import pandas as pd
import numpy as np

dataset_path = "/Volumes/KINGSTON/NeuroCore/neurocore_lab/data/EEG/TDCS_ds006126"
output_folder = os.path.join(dataset_path, "Output")
os.makedirs(output_folder, exist_ok=True)
output_csv = os.path.join(output_folder, "TDCS_MI_superfirma_debug_recursive.csv")

csv_rows = []

print("=== DEBUG GENERAZIONE CSV TDCS-MI (RICORSIVO) ===")

# Scansione ricorsiva
for root, dirs, files in os.walk(dataset_path):
    vhdr_files = [f for f in files if f.endswith('.vhdr')]
    if not vhdr_files:
        continue

    for vhdr_file in vhdr_files:
        raw_path = os.path.join(root, vhdr_file)
        subj_folder = os.path.basename(root)  # prendi la sottocartella corrente
        try:
            raw = mne.io.read_raw_brainvision(raw_path, preload=True, verbose=False)
        except Exception as e:
            print(f"[ERROR] {vhdr_file}: {e}")
            continue
        raw.filter(1,45,fir_design='firwin', verbose=False)
        events, event_id = mne.events_from_annotations(raw, verbose=False)

        for cond_name, cond_code in event_id.items():
            epochs = mne.Epochs(raw, events, event_id={cond_name:cond_code}, tmin=0, tmax=2,
                                baseline=None, preload=True, verbose=False)
            data = epochs.get_data()
            n_epochs, n_ch, n_times = data.shape if data.size>0 else (0,0,0)

            if n_epochs==0:
                print(f"[SKIP] {subj_folder} - {cond_name} -> nessuna epoca valida")
                continue

            print(f"[OK] {subj_folder} - {cond_name} -> epoche: {n_epochs}, canali: {n_ch}, punti temporali: {n_times}")

            # Scrive almeno qualche feature semplice come test
            for ch_idx, ch_name in enumerate(raw.info['ch_names']):
                sig_epochs = data[:,ch_idx,:]
                sig_flat = sig_epochs.reshape(-1)
                mean_val = np.mean(sig_flat)
                var_val = np.var(sig_flat)
                csv_rows.append({'subject':subj_folder,'condition':cond_name,'channel':ch_name,
                                 'mean':mean_val,'variance':var_val})

# Salvataggio CSV debug
if csv_rows:
    df_debug = pd.DataFrame(csv_rows)
    df_debug.to_csv(output_csv,index=False)
    print(f"✅ CSV debug ricorsivo salvato in: {output_csv}")
else:
    print("❌ Nessuna epoca valida trovata in tutto il dataset, CSV vuoto")

=== DEBUG GENERAZIONE CSV TDCS-MI (RICORSIVO) ===
❌ Nessuna epoca valida trovata in tutto il dataset, CSV vuoto


In [19]:
!pip install mne-bids

In [20]:
from pathlib import Path
import numpy as np
import pandas as pd
import mne
from mne_bids import BIDSPath, read_raw_bids
import warnings

warnings.filterwarnings("ignore", category=RuntimeWarning)

BIDS_ROOT = Path("/Volumes/KINGSTON/NeuroCore/neurocore_lab/data/EEG/TDCS_ds006126")
SEGMENT_S = 60
WIN_S = 1.0
FMIN, FMAX = 1., 40.

subjects = sorted([p.name.replace("sub-","") for p in BIDS_ROOT.glob("sub-*")])
print("Soggetti trovati:", len(subjects))

results = []
errors = []

for i, sub in enumerate(subjects):
    try:
        bids_path = BIDSPath(
            subject=sub,
            task=None,
            datatype="eeg",
            root=BIDS_ROOT
        )

        raw = read_raw_bids(bids_path, verbose=False)

        raw.pick_types(eeg=True)
        raw.filter(FMIN, FMAX, verbose=False)

        data = raw.get_data()
        sfreq = raw.info["sfreq"]

        seg_samples = int(SEGMENT_S * sfreq)
        win_samples = int(WIN_S * sfreq)

        if data.shape[1] < seg_samples:
            continue

        segment = data[:, :seg_samples]
        n_windows = seg_samples // win_samples

        values = []
        for w in range(n_windows):
            s = w * win_samples
            t = s + win_samples
            x = segment[:, s:t]
            values.append(np.max(np.abs(x)))

        results.append({
            "subject": sub,
            "p95_win_maxabs": float(np.percentile(values, 95))
        })

        if i % 20 == 0:
            print(f"{i}/{len(subjects)}")

    except Exception as e:
        errors.append((sub, str(e)))

df = pd.DataFrame(results)

print("\nOK:", len(df), "| Errori:", len(errors))

if len(df):
    df = df.sort_values("p95_win_maxabs", ascending=False)
    print("\nTOP 15:")
    print(df.head(15))

df.to_csv(BIDS_ROOT / "neurocore_batch_results.csv", index=False)
print("\nSalvato.")


Soggetti trovati: 5

OK: 0 | Errori: 5

Salvato.


In [21]:
import sys
!{sys.executable} -m pip install --upgrade mne-bids

In [22]:
from pathlib import Path
import numpy as np
import pandas as pd
import mne
import warnings
from mne_bids import BIDSPath, read_raw_bids

# ======================
# CONFIG
# ======================
BIDS_ROOT = Path("/Volumes/KINGSTON/NeuroCore/neurocore_lab/data/EEG/TDCS_ds006126")
SEGMENT_S = 60
WIN_S = 1.0
FMIN, FMAX = 1., 40.
# ======================

assert BIDS_ROOT.exists(), f"Percorso non trovato: {BIDS_ROOT}"
warnings.filterwarnings("ignore", category=RuntimeWarning)

subjects = sorted([p.name.replace("sub-","") for p in BIDS_ROOT.glob("sub-*") if p.is_dir()])
print("Soggetti trovati:", len(subjects))

def guess_task(sub):
    eeg_dir = BIDS_ROOT / f"sub-{sub}" / "eeg"
    edfs = sorted(eeg_dir.glob("*_eeg.edf"))
    if not edfs:
        return None
    name = edfs[0].name
    if "_task-" in name:
        return name.split("_task-")[1].split("_")[0]
    return None

def compute_metric(raw):
    raw.load_data()
    try:
        raw.pick_types(eeg=True)
        if len(raw.ch_names) == 0:
            raw.pick(raw.ch_names)
    except:
        raw.pick(raw.ch_names)

    try:
        raw.filter(FMIN, FMAX, verbose=False)
        filtered = True
    except:
        filtered = False

    data = raw.get_data()
    sfreq = raw.info["sfreq"]

    seg_samples = int(SEGMENT_S * sfreq)
    win_samples = int(WIN_S * sfreq)

    if data.shape[1] < seg_samples:
        return None

    segment = data[:, :seg_samples]
    n_windows = seg_samples // win_samples

    win_maxabs = []
    for w in range(n_windows):
        s = w * win_samples
        t = s + win_samples
        x = segment[:, s:t]
        win_maxabs.append(float(np.max(np.abs(x))))

    return {
        "p95_win_maxabs": float(np.percentile(win_maxabs, 95)),
        "median_win_maxabs": float(np.median(win_maxabs)),
        "max_win_maxabs": float(np.max(win_maxabs)),
        "sfreq": sfreq,
        "n_channels": segment.shape[0],
        "filtered": filtered
    }

results = []
errors = []

for i, sub in enumerate(subjects, 1):
    try:
        task = guess_task(sub)
        bids_path = BIDSPath(subject=sub, task=task, datatype="eeg", root=BIDS_ROOT)
        raw = read_raw_bids(bids_path, verbose=False)

        metric = compute_metric(raw)
        if metric is None:
            errors.append({"subject": sub, "error": "too_short"})
            continue

        metric["subject"] = sub
        metric["task"] = task
        results.append(metric)

        if i % 10 == 0:
            print(f"[{i}/{len(subjects)}] OK={len(results)} ERR={len(errors)}")

    except Exception as e:
        errors.append({"subject": sub, "error": str(e)})

df = pd.DataFrame(results)
df_err = pd.DataFrame(errors)

print("\n=== RISULTATI ===")
print("OK:", len(df), "| Errori:", len(df_err))

df.to_csv(BIDS_ROOT / "neurocore_batch_results.csv", index=False)
df_err.to_csv(BIDS_ROOT / "neurocore_batch_errors.csv", index=False)

print("Salvato CSV completo.")

Soggetti trovati: 5

=== RISULTATI ===
OK: 0 | Errori: 5
Salvato CSV completo.


In [23]:
from pathlib import Path
import numpy as np
import pandas as pd
import mne
import warnings
from mne_bids import BIDSPath, read_raw_bids

# ======================
# CONFIG
# ======================
BIDS_ROOT = Path("/Volumes/KINGSTON/NeuroCore/neurocore_lab/data/EEG/TDCS_ds006126")
SEGMENT_S = 60
WIN_S = 1.0
FMIN, FMAX = 1., 40.
# ======================

assert BIDS_ROOT.exists(), f"Percorso non trovato: {BIDS_ROOT}"
warnings.filterwarnings("ignore", category=RuntimeWarning)

subjects = sorted([p.name.replace("sub-","") for p in BIDS_ROOT.glob("sub-*") if p.is_dir()])
print("Soggetti trovati:", len(subjects))

def guess_task(sub):
    eeg_dir = BIDS_ROOT / f"sub-{sub}" / "eeg"
    # Ricerca ricorsiva nelle sottocartelle
    edfs = sorted(eeg_dir.rglob("*_eeg.edf"))
    if not edfs:
        return None
    name = edfs[0].name
    if "_task-" in name:
        return name.split("_task-")[1].split("_")[0]
    return None

def compute_metric(raw):
    raw.load_data()
    try:
        raw.pick_types(eeg=True)
        if len(raw.ch_names) == 0:
            raw.pick(raw.ch_names)
    except:
        raw.pick(raw.ch_names)

    try:
        raw.filter(FMIN, FMAX, verbose=False)
        filtered = True
    except:
        filtered = False

    data = raw.get_data()
    sfreq = raw.info["sfreq"]

    seg_samples = int(SEGMENT_S * sfreq)
    win_samples = int(WIN_S * sfreq)

    if data.shape[1] < seg_samples:
        return None

    segment = data[:, :seg_samples]
    n_windows = seg_samples // win_samples

    win_maxabs = []
    for w in range(n_windows):
        s = w * win_samples
        t = s + win_samples
        x = segment[:, s:t]
        win_maxabs.append(float(np.max(np.abs(x))))

    return {
        "p95_win_maxabs": float(np.percentile(win_maxabs, 95)),
        "median_win_maxabs": float(np.median(win_maxabs)),
        "max_win_maxabs": float(np.max(win_maxabs)),
        "sfreq": sfreq,
        "n_channels": segment.shape[0],
        "filtered": filtered
    }

results = []
errors = []

for i, sub in enumerate(subjects, 1):
    try:
        task = guess_task(sub)
        bids_path = BIDSPath(subject=sub, task=task, datatype="eeg", root=BIDS_ROOT)
        raw = read_raw_bids(bids_path, verbose=False)

        metric = compute_metric(raw)
        if metric is None:
            errors.append({"subject": sub, "error": "too_short"})
            continue

        metric["subject"] = sub
        metric["task"] = task
        results.append(metric)

        if i % 10 == 0:
            print(f"[{i}/{len(subjects)}] OK={len(results)} ERR={len(errors)}")

    except Exception as e:
        errors.append({"subject": sub, "error": str(e)})

df = pd.DataFrame(results)
df_err = pd.DataFrame(errors)

print("\n=== RISULTATI ===")
print("OK:", len(df), "| Errori:", len(df_err))

df.to_csv(BIDS_ROOT / "neurocore_batch_results.csv", index=False)
df_err.to_csv(BIDS_ROOT / "neurocore_batch_errors.csv", index=False)

print("Salvato CSV completo.")

Soggetti trovati: 5

=== RISULTATI ===
OK: 0 | Errori: 5
Salvato CSV completo.


In [24]:
from pathlib import Path
import numpy as np
import pandas as pd
import mne
import warnings
from mne_bids import BIDSPath, read_raw_bids

# ======================
# CONFIG
# ======================
BIDS_ROOT = Path("/Volumes/KINGSTON/NeuroCore/neurocore_lab/data/EEG/TDCS_ds006126")
SEGMENT_S = 60
WIN_S = 1.0
FMIN, FMAX = 1., 40.
# ======================

warnings.filterwarnings("ignore", category=RuntimeWarning)

assert BIDS_ROOT.exists(), f"Percorso non trovato: {BIDS_ROOT}"

# Trova soggetti
subjects = sorted([p.name.replace("sub-","") for p in BIDS_ROOT.glob("sub-*") if p.is_dir()])
print("Soggetti trovati:", subjects)

# Funzione per identificare task
def guess_task(sub):
    eeg_dir = BIDS_ROOT / f"sub-{sub}" / "eeg"
    # ricerca ricorsiva
    edfs = sorted(eeg_dir.rglob("*_eeg.edf"))
    if not edfs:
        return None
    name = edfs[0].name
    if "_task-" in name:
        return name.split("_task-")[1].split("_")[0]
    return None

# Funzione di metriche robuste
def compute_metric(raw):
    try:
        raw.load_data()
    except:
        return None

    try:
        raw.pick_types(eeg=True)
        if len(raw.ch_names) == 0:
            return None
    except:
        return None

    try:
        raw.filter(FMIN, FMAX, verbose=False)
        filtered = True
    except:
        filtered = False

    data = raw.get_data()
    sfreq = raw.info.get("sfreq", np.nan)

    if data.shape[1] < SEGMENT_S * sfreq:
        return None

    seg_samples = int(SEGMENT_S * sfreq)
    win_samples = int(WIN_S * sfreq)
    segment = data[:, :seg_samples]
    n_windows = max(seg_samples // win_samples, 1)

    win_maxabs = []
    for w in range(n_windows):
        s = w * win_samples
        t = min(s + win_samples, segment.shape[1])
        x = segment[:, s:t]
        win_maxabs.append(float(np.max(np.abs(x))))

    return {
        "p95_win_maxabs": float(np.percentile(win_maxabs, 95)),
        "median_win_maxabs": float(np.median(win_maxabs)),
        "max_win_maxabs": float(np.max(win_maxabs)),
        "sfreq": sfreq,
        "n_channels": segment.shape[0],
        "filtered": filtered
    }

results = []
errors = []

for i, sub in enumerate(subjects, 1):
    try:
        task = guess_task(sub)
        if task is None:
            errors.append({"subject": sub, "error": "no_task_found"})
            continue

        bids_path = BIDSPath(subject=sub, task=task, datatype="eeg", root=BIDS_ROOT)

        # PROVA a leggere qualsiasi raw, ignora header strani
        try:
            raw = read_raw_bids(bids_path, verbose=False)
        except Exception as e:
            errors.append({"subject": sub, "error": f"read_raw_bids_fail:{e}"})
            continue

        metric = compute_metric(raw)
        if metric is None:
            errors.append({"subject": sub, "error": "metric_fail"})
            continue

        metric["subject"] = sub
        metric["task"] = task
        results.append(metric)

        print(f"[{i}/{len(subjects)}] OK={len(results)} ERR={len(errors)}")

    except Exception as e:
        errors.append({"subject": sub, "error": str(e)})

# Salvataggio CSV robusto
df = pd.DataFrame(results)
df_err = pd.DataFrame(errors)

df.to_csv(BIDS_ROOT / "neurocore_batch_results_robust.csv", index=False)
df_err.to_csv(BIDS_ROOT / "neurocore_batch_errors_robust.csv", index=False)

print("\n=== RISULTATI ===")
print("Soggetti OK:", len(df))
print("Soggetti con errori:", len(df_err))
print("CSV salvati.")

Soggetti trovati: ['AnSt1', 'FeKl03', 'IrCh04', 'Kiko09', 'SoNi11']

=== RISULTATI ===
Soggetti OK: 0
Soggetti con errori: 5
CSV salvati.


In [25]:
from pathlib import Path
import numpy as np
import pandas as pd
import mne
import warnings

# ======================
# CONFIG
# ======================
BIDS_ROOT = Path("/Volumes/KINGSTON/NeuroCore/neurocore_lab/data/EEG/TDCS_ds006126")
SEGMENT_S = 10       # lunghezza minima segmento ridotta per catturare più dati
WIN_S = 1.0
FMIN, FMAX = 1., 40.
# ======================

warnings.filterwarnings("ignore", category=RuntimeWarning)

assert BIDS_ROOT.exists(), f"Percorso non trovato: {BIDS_ROOT}"

results = []
errors = []

# Scansiona tutti i soggetti e i file edf ricorsivamente
for sub_path in sorted(BIDS_ROOT.glob("sub-*")):
    sub = sub_path.name.replace("sub-","")
    eeg_dir = sub_path / "eeg"

    # Ricerca tutti i file *_eeg.edf anche nelle sottocartelle
    edf_files = list(eeg_dir.rglob("*_eeg.edf"))
    if not edf_files:
        errors.append({"subject": sub, "file": None, "error": "no_edf_found"})
        continue

    for edf_file in edf_files:
        try:
            raw = mne.io.read_raw_edf(edf_file, preload=True, verbose=False)
            raw.pick_types(eeg=True)

            if len(raw.ch_names) == 0 or raw.n_times < SEGMENT_S * raw.info["sfreq"]:
                errors.append({"subject": sub, "file": str(edf_file), "error": "too_short_or_no_channels"})
                continue

            # Filtro robusto
            try:
                raw.filter(FMIN, FMAX, verbose=False)
                filtered = True
            except:
                filtered = False

            data = raw.get_data()
            sfreq = raw.info.get("sfreq", np.nan)
            seg_samples = int(min(SEGMENT_S*sfreq, data.shape[1]))
            win_samples = int(WIN_S*sfreq)
            n_windows = max(seg_samples // win_samples, 1)

            win_maxabs = []
            segment = data[:, :seg_samples]

            for w in range(n_windows):
                s = w*win_samples
                t = min(s + win_samples, segment.shape[1])
                x = segment[:, s:t]
                win_maxabs.append(float(np.max(np.abs(x))))

            results.append({
                "subject": sub,
                "file": str(edf_file.relative_to(BIDS_ROOT)),
                "p95_win_maxabs": float(np.percentile(win_maxabs, 95)),
                "median_win_maxabs": float(np.median(win_maxabs)),
                "max_win_maxabs": float(np.max(win_maxabs)),
                "sfreq": sfreq,
                "n_channels": segment.shape[0],
                "filtered": filtered
            })

            print(f"[OK] {sub} | {edf_file.name}")

        except Exception as e:
            errors.append({"subject": sub, "file": str(edf_file), "error": str(e)})

# Salvataggio CSV finale
df = pd.DataFrame(results)
df_err = pd.DataFrame(errors)

df.to_csv(BIDS_ROOT / "neurocore_all_edf_results.csv", index=False)
df_err.to_csv(BIDS_ROOT / "neurocore_all_edf_errors.csv", index=False)

print("\n=== RISULTATI ===")
print("File OK:", len(df))
print("File con errori:", len(df_err))
print("CSV salvati nella root del dataset.")


=== RISULTATI ===
File OK: 0
File con errori: 5
CSV salvati nella root del dataset.
